# Prepare the merged training dataset

Wraps `scripts/prepare_dataset.py` with intermediate inspection. Same pipeline (normalize → dedupe → group-aware stratified split → source-weight downsample → write).

Run after `scripts/download_datasets.py` has populated `data/raw/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

## Option A — fast first pass

Skip dedup and source weights. Fast (~60 s on 30k images). Use this to verify the pipeline runs end-to-end before committing to the slow dedup pass.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'prepare_dataset.py'),
                '--no-dedup', '--no-weights',
                '--train', '0.8', '--val', '0.1', '--test', '0.1'], check=True)

## Option B — full pipeline

Includes perceptual-hash dedup (cross-source) and per-source train weights from `sources.yaml`. Takes a few minutes per 10k images.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'prepare_dataset.py'),
                '--train', '0.8', '--val', '0.1', '--test', '0.1'], check=True)

## Inspect the result

In [ ]:
import yaml
MERGED = ROOT / 'data' / 'merged'
print((MERGED / 'PREP_REPORT.md').read_text())

In [ ]:
with open(MERGED / 'data.yaml') as f:
    print(f.read())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'inspect_dataset.py'),
                '--dataset', str(MERGED)], check=True)
from PIL import Image
import matplotlib.pyplot as plt
for split in ('train', 'val', 'test'):
    p = MERGED / f'sample_{split}.png'
    if p.exists():
        plt.figure(figsize=(12, 12)); plt.imshow(Image.open(p)); plt.axis('off')
        plt.title(split); plt.show()